In [ ]:
import os
import cv2
import random
import numpy as np
from scipy.fftpack import dct
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt
import seaborn as sns
from joblib import Parallel, delayed

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout, BatchNormalization, Reshape
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report, roc_curve, auc, roc_auc_score

np.random.seed(42)
tf.random.set_seed(42)
random.seed(42)
print("TensorFlow Version:", tf.__version__)

In [ ]:

valid_exts = ('.jpg', '.jpeg', '.png', '.tif', '.tiff')

# =========================
# Danh sách tất cả dataset
# =========================

authentic_dirs = [

    # CASIA
    '/kaggle/input/datasets/divg07/casia-20-image-tampering-detection-dataset/CASIA2/Au',

    # Columbia
    '/kaggle/input/datasets/shriya0/columbia/Columbia Uncompressed Image Splicing Detection/4cam_auth/4cam_auth',


    # IMD2020
    '/kaggle/input/datasets/saikotahmed/imd2020/IMD2020/Authentic'

]


tampering_dirs = [

    # CASIA
    '/kaggle/input/datasets/divg07/casia-20-image-tampering-detection-dataset/CASIA2/Tp',

    # Columbia
    '/kaggle/input/datasets/shriya0/columbia/Columbia Uncompressed Image Splicing Detection/4cam_splc/4cam_splc',


    # IMD2020
    '/kaggle/input/datasets/saikotahmed/imd2020/IMD2020/img'


]

# =========================
# Hàm lấy tất cả ảnh
# =========================

def collect_image_paths(dirs):
    paths = []

    for d in dirs:

        if not os.path.exists(d):
            print(f"[WARNING] Không tìm thấy thư mục: {d}")
            continue

        files = [
            os.path.join(d, f)
            for f in os.listdir(d)
            if f.lower().endswith(valid_exts)
        ]

        print(f"{d} → {len(files)} ảnh")

        paths.extend(files)

    return paths


# =========================
# Thu thập dữ liệu
# =========================

all_authentic_paths = collect_image_paths(authentic_dirs)

all_tampering_paths = collect_image_paths(tampering_dirs)

# CÂN BẰNG DỮ LIỆU
num_authentic = len(all_authentic_paths)
num_tampering = len(all_tampering_paths)

# if num_authentic > num_tampering:
#     # Nếu Gốc nhiều hơn -> Lấy mẫu ngẫu nhiên Gốc cho bằng Giả
#     authentic_paths = random.sample(all_authentic_paths, num_tampering)
#     tampering_paths = all_tampering_paths
# elif num_tampering > num_authentic:
#     # Nếu Giả nhiều hơn -> Lấy mẫu ngẫu nhiên Giả cho bằng Gốc
#     tampering_paths = random.sample(all_tampering_paths, num_authentic)
#     authentic_paths = all_authentic_paths
# else:
#     # Nếu bằng nhau -> Giữ nguyên cả hai
#     authentic_paths = all_authentic_paths
#     tampering_paths = all_tampering_paths

authentic_paths = all_authentic_paths
tampering_paths = all_tampering_paths
print(f"[Raw] Ảnh Gốc (Nhãn 0) : {len(all_authentic_paths)}")
print(f"[Raw] Ảnh Giả (Nhãn 1) : {len(all_tampering_paths)}")
print(f"[ĐÃ CÂN BẰNG] Ảnh Gốc (Nhãn 0) : {len(authentic_paths)}")
print(f"[ĐÃ CÂN BẰNG] Ảnh Giả (Nhãn 1) : {len(tampering_paths)}")

In [ ]:
# def get_tpm_matrix(diff_mat, T, direction):
#     diff_mat = np.clip(diff_mat, -T, T).astype(int) + T
#     dim = 2 * T + 1
#     tpm = np.zeros((dim, dim))
#     h, w = diff_mat.shape
    
#     if direction == 'horizontal':
#         for i in range(h):
#             for j in range(w - 1):
#                 tpm[diff_mat[i, j], diff_mat[i, j+1]] += 1
#     elif direction == 'vertical':
#         for i in range(h - 1):
#             for j in range(w):
#                 tpm[diff_mat[i, j], diff_mat[i+1, j]] += 1
                
#     row_sums = tpm.sum(axis=1, keepdims=True)
#     row_sums[row_sums == 0] = 1 
#     tpm = tpm / row_sums
#     return tpm.flatten()

# def extract_markov_dct_features(image_path, block_size=8, T=4):
#     img = cv2.imread(image_path)
#     if img is None: return None
    
#     img_ycbcr = cv2.cvtColor(img, cv2.COLOR_BGR2YCrCb)
#     all_channel_features = []
    
#     for channel in range(3):
#         comp = img_ycbcr[:, :, channel]
#         h, w = comp.shape
#         h, w = h - (h % block_size), w - (w % block_size)
#         comp = comp[:h, :w]
        
#         # BDCT
#         dct_img = np.zeros_like(comp, dtype=np.float32)
#         for i in range(0, h, block_size):
#             for j in range(0, w, block_size):
#                 block = comp[i:i+block_size, j:j+block_size]
#                 dct_img[i:i+block_size, j:j+block_size] = dct(dct(block.T, norm='ortho').T, norm='ortho')
        
#         # Giải tương quan 4 hướng
#         diff_intra_h = dct_img[:, :-1] - dct_img[:, 1:]
#         diff_intra_v = dct_img[:-1, :] - dct_img[1:, :]
#         diff_inter_h = dct_img[:, :-block_size] - dct_img[:, block_size:]
#         diff_inter_v = dct_img[:-block_size, :] - dct_img[block_size:, :]
        
#         # 8 TPMs mỗi kênh
#         tpm_intra_h_horiz = get_tpm_matrix(diff_intra_h, T, 'horizontal')
#         tpm_intra_h_vert  = get_tpm_matrix(diff_intra_h, T, 'vertical')
#         tpm_intra_v_horiz = get_tpm_matrix(diff_intra_v, T, 'horizontal')
#         tpm_intra_v_vert  = get_tpm_matrix(diff_intra_v, T, 'vertical')
#         tpm_inter_h_horiz = get_tpm_matrix(diff_inter_h, T, 'horizontal')
#         tpm_inter_h_vert  = get_tpm_matrix(diff_inter_h, T, 'vertical')
#         tpm_inter_v_horiz = get_tpm_matrix(diff_inter_v, T, 'horizontal')
#         tpm_inter_v_vert  = get_tpm_matrix(diff_inter_v, T, 'vertical')
        
#         channel_vector = np.concatenate([
#             tpm_intra_h_horiz, tpm_intra_h_vert, tpm_intra_v_horiz, tpm_intra_v_vert,
#             tpm_inter_h_horiz, tpm_inter_h_vert, tpm_inter_v_horiz, tpm_inter_v_vert
#         ])
#         all_channel_features.append(channel_vector)
    
#     return np.concatenate(all_channel_features) # Vector 1944 chiều

# def process_single_image(path, label):
#     feat = extract_markov_dct_features(path)
#     if feat is not None: return feat, label
#     return None

In [ ]:
import cv2
import numpy as np
from scipy.fftpack import dct

def get_tpm_matrix(diff_mat, T, direction):
    
    diff_mat = np.clip(diff_mat, -T, T).astype(int) + T
    dim = 2 * T + 1
    
    # 1. Kéo phẳng ma trận theo hướng cần xét
    if direction == 'horizontal':
        current_states = diff_mat[:, :-1].flatten()
        next_states = diff_mat[:, 1:].flatten()
    elif direction == 'vertical':
        current_states = diff_mat[:-1, :].flatten()
        next_states = diff_mat[1:, :].flatten()
        
    # 2. Chuyển đổi tọa độ 2D thành Index 1D
    linear_indices = current_states * dim + next_states
    
    # 3. Đếm tần suất xuất hiện cực nhanh bằng C-backend
    tpm_1d = np.bincount(linear_indices, minlength=dim*dim)
    
    # 4. Định hình lại thành ma trận 2D
    tpm = tpm_1d.reshape((dim, dim)).astype(float)
    
    # 5. Chuẩn hóa xác suất
    row_sums = tpm.sum(axis=1, keepdims=True)
    row_sums[row_sums == 0] = 1 
    tpm = tpm / row_sums
    
    return tpm.flatten()

def extract_markov_dct_features(image_path, block_size=8, T=4):
    img = cv2.imread(image_path)
    if img is None: 
        return None
    
    img_ycbcr = cv2.cvtColor(img, cv2.COLOR_BGR2YCrCb)
    all_channel_features = []
    
    for channel in range(3):
        # Ép kiểu float32 ngay từ đầu để chuẩn bị cho DCT
        comp = img_ycbcr[:, :, channel].astype(np.float32) 
        h, w = comp.shape
        h, w = h - (h % block_size), w - (w % block_size)
        comp = comp[:h, :w]
        
        # ==============================================================
        # KỸ THUẬT: BLOCK DCT VECTOR HÓA BẰNG RESHAPE 4D
        # ==============================================================
        # Chia ma trận 2D thành lưới các block 8x8 (Tạo tensor 4D)
        blocks = comp.reshape(h // block_size, block_size, w // block_size, block_size)
        blocks = blocks.transpose(0, 2, 1, 3) # Kích thước lúc này: (Số block dọc, Số block ngang, 8, 8)
        
        # Áp dụng DCT trực tiếp lên toàn bộ các block cùng lúc (axis=-1 là hàng, axis=-2 là cột)
        dct_blocks = dct(dct(blocks, axis=-1, norm='ortho'), axis=-2, norm='ortho')
        
        # Lắp ghép các block 8x8 trả lại thành bức ảnh 2D ban đầu
        dct_img = dct_blocks.transpose(0, 2, 1, 3).reshape(h, w)
        # ==============================================================
        
        # Giải tương quan 4 hướng (Đoạn này bạn viết bằng slicing đã rất tối ưu rồi)
        diff_intra_h = dct_img[:, :-1] - dct_img[:, 1:]
        diff_intra_v = dct_img[:-1, :] - dct_img[1:, :]
        diff_inter_h = dct_img[:, :-block_size] - dct_img[:, block_size:]
        diff_inter_v = dct_img[:-block_size, :] - dct_img[block_size:, :]
        
        # 8 TPMs mỗi kênh
        tpm_intra_h_horiz = get_tpm_matrix(diff_intra_h, T, 'horizontal')
        tpm_intra_h_vert  = get_tpm_matrix(diff_intra_h, T, 'vertical')
        tpm_intra_v_horiz = get_tpm_matrix(diff_intra_v, T, 'horizontal')
        tpm_intra_v_vert  = get_tpm_matrix(diff_intra_v, T, 'vertical')
        tpm_inter_h_horiz = get_tpm_matrix(diff_inter_h, T, 'horizontal')
        tpm_inter_h_vert  = get_tpm_matrix(diff_inter_h, T, 'vertical')
        tpm_inter_v_horiz = get_tpm_matrix(diff_inter_v, T, 'horizontal')
        tpm_inter_v_vert  = get_tpm_matrix(diff_inter_v, T, 'vertical')
        
        channel_vector = np.concatenate([
            tpm_intra_h_horiz, tpm_intra_h_vert, tpm_intra_v_horiz, tpm_intra_v_vert,
            tpm_inter_h_horiz, tpm_inter_h_vert, tpm_inter_v_horiz, tpm_inter_v_vert
        ])
        all_channel_features.append(channel_vector)
    
    return np.concatenate(all_channel_features) # Trả về Vector 1944 chiều

def process_single_image(path, label):
    feat = extract_markov_dct_features(path)
    if feat is not None: 
        return feat, label
    return None

In [ ]:
print("Đang xử lý ảnh Gốc (Authentic)...")
results_auth = Parallel(n_jobs=-1, backend="multiprocessing")(
    delayed(process_single_image)(path, 0) for path in tqdm(authentic_paths)
)

print("Đang xử lý ảnh Giả mạo (tampering)...")
results_tampering = Parallel(n_jobs=-1, backend="multiprocessing")(
    delayed(process_single_image)(path, 1) for path in tqdm(tampering_paths)
)

all_results = results_auth + results_tampering
valid_results = [res for res in all_results if res is not None]

X = np.array([res[0] for res in valid_results])
y = np.array([res[1] for res in valid_results])

np.save('X_binary.npy', X)
np.save('y_binary.npy', y)
print(f"\n[HOÀN TẤT] Kích thước mảng X: {X.shape}, mảng y: {y.shape}")

In [ ]:
X = np.load('X_binary.npy')
y = np.load('y_binary.npy')

# Chia 80/20
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(f"Tập Train: {X_train.shape[0]} mẫu | Tập Test: {X_test.shape[0]} mẫu")

def build_binary_cnn(input_shape, T=4):
    dim = 2 * T + 1 
    channels = 24 # 24 ma trận TPM
    
    model = Sequential([
        Reshape((dim, dim, channels), input_shape=input_shape),
        Conv2D(32, (3, 3), activation='relu', padding='same'),
        BatchNormalization(),
        MaxPooling2D((2, 2)),
        Conv2D(64, (2, 2), activation='relu', padding='same'),
        BatchNormalization(),
        Flatten(),
        Dense(128, activation='relu'),
        Dropout(0.5),
        Dense(1, activation='sigmoid') # Nhị phân: 0 (Gốc), 1 (Giả)
    ])
    return model

model = build_binary_cnn(input_shape=(X_train.shape[1],))
model.summary()

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint

# Định nghĩa file lưu model
checkpoint_filepath = 'best_model_weights.keras'

# Cấu hình Checkpoint
model_checkpoint = ModelCheckpoint(
    filepath=checkpoint_filepath,
    monitor='val_loss',      # Theo dõi giá trị loss trên tập validation
    save_best_only=True,     # CHỈ LƯU khi kết quả tốt hơn các epoch trước đó
    mode='min',              # Lưu khi val_loss đạt giá trị nhỏ nhất (min)
    verbose=1                # In thông báo mỗi khi model được lưu đè
)

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='binary_crossentropy',
    metrics=['accuracy', tf.keras.metrics.AUC(name='auc')]
)

# Thêm model_checkpoint vào list callbacks
callbacks = [
    EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=4),
    model_checkpoint # <--- Thêm dòng này
]

# Huấn luyện bình thường
history = model.fit(
    X_train, y_train,
    validation_split=0.2, 
    epochs=200,
    batch_size=32,
    callbacks=callbacks,
    verbose=1
)

In [ ]:
# Dự đoán trên tập Test
y_pred_prob = model.predict(X_test).ravel()
y_pred = (y_pred_prob > 0.5).astype(int)

# 1. BIỂU ĐỒ QUÁ TRÌNH HUẤN LUYỆN
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
axes[0].plot(history.history['loss'], label='Train Loss', lw=2)
axes[0].plot(history.history['val_loss'], label='Val Loss', linestyle='--', lw=2)
axes[0].set_title('Loss')
axes[0].legend(); axes[0].grid(True, linestyle=':', alpha=0.7)

axes[1].plot(history.history['accuracy'], label='Train Acc', lw=2)
axes[1].plot(history.history['val_accuracy'], label='Val Acc', linestyle='--', lw=2)
axes[1].set_title('Accuracy')
axes[1].legend(); axes[1].grid(True, linestyle=':', alpha=0.7)

axes[2].plot(history.history['auc'], label='Train AUC', lw=2)
axes[2].plot(history.history['val_auc'], label='Val AUC', linestyle='--', lw=2)
axes[2].set_title('AUC')
axes[2].legend(); axes[2].grid(True, linestyle=':', alpha=0.7)
plt.show()

# 2. ROC CURVE & CONFUSION MATRIX
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# ROC
fpr, tpr, _ = roc_curve(y_test, y_pred_prob)
roc_auc = auc(fpr, tpr)
axes[0].plot(fpr, tpr, color='darkorange', lw=2, label=f'AUC = {roc_auc:.4f}')
axes[0].plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
axes[0].set_title('ROC Curve (Authentic vs tampering)', fontweight='bold')
axes[0].legend(loc="lower right"); axes[0].grid(True, linestyle=':', alpha=0.7)

# CM
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False,
            xticklabels=['Authentic (0)', 'tampering (1)'],
            yticklabels=['Authentic (0)', 'tampering (1)'], ax=axes[1], annot_kws={"size": 15})
axes[1].set_title('Confusion Matrix', fontweight='bold')
plt.show()

# 3. TEXT REPORT
print("="*60)
print("     CLASSIFICATION REPORT (BINARY: AUTHENTIC VS tampering)     ")
print("="*60)
print(classification_report(y_test, y_pred, target_names=['Authentic (0)', 'tampering (1)']))
print("="*60)

In [ ]:
import matplotlib.pyplot as plt
import cv2
from tensorflow.keras.models import load_model

# 1. Load mô hình
loaded_model = load_model('best_model_weights.keras')

idx_authentic = np.where(y_test == 0)[0]
idx_tampering = np.where(y_test == 1)[0]

sample_auth = np.random.choice(idx_authentic, 1, replace=False)
sample_tampering = np.random.choice(idx_tampering, 11, replace=False)
test_samples = np.concatenate([sample_auth, sample_tampering])

# 3. Thiết lập khung hình hiển thị (2 hàng, 3 cột)
plt.figure(figsize=(18, 10))

for i, idx in enumerate(test_samples):
    # --- DỰ ĐOÁN ---
    feature_vector = X_test[idx].reshape(1, -1)
    prob = loaded_model.predict(feature_vector, verbose=0)[0][0]
    pred_label = 1 if prob > 0.5 else 0
    
    # --- ĐỌC ẢNH GỐC ĐỂ HIỂN THỊ ---
    if y_test[idx] == 0:
        img_path = random.choice(authentic_paths) 
    else:
        img_path = random.choice(tampering_paths)
        
    img = cv2.imread(img_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    
    # --- VẼ ---
    plt.subplot(4, 6, i + 1)
    plt.imshow(img)
    
    is_correct = (pred_label == y_test[idx])
    color = 'green' if is_correct else 'red'
    
    true_str = "AUTH" if y_test[idx] == 0 else "tampering"
    pred_str = "AUTH" if pred_label == 0 else "tampering"
    
    plt.title(f"Thực tế: {true_str}\nDự đoán: {pred_str}\nProb: {prob:.4f}", 
              color=color, fontsize=12, fontweight='bold')
    plt.axis('off')

plt.tight_layout()
plt.show()

In [ ]:
def predict_external_image(url, model):
    temp_path = "inference_sample.jpg"
    
    try:
        response = requests.get(url, timeout=15)
        if response.status_code != 200:
            print(f" Không thể tải ảnh từ: {url}")
            return None, None
            
        with open(temp_path, "wb") as f:
            f.write(response.content)
            
        img_cv2 = cv2.imread(temp_path)
        
        if img_cv2 is None:
            print(f" OpenCV lỗi file từ: {url}")
            return None, None

        # 4. Trích xuất đặc trưng Markov DCT từ FILE PATH
        feat = extract_markov_dct_features(temp_path) 
        
        if feat is not None:
            # 5. Thực hiện dự đoán (Inference)
            feature_vector = feat.reshape(1, -1)
            prob = model.predict(feature_vector, verbose=0)[0][0]
            
            # Chuyển hệ màu để hiển thị đúng trên Matplotlib (BGR -> RGB)
            img_display = cv2.cvtColor(img_cv2, cv2.COLOR_BGR2RGB)
            return img_display, prob
            
    except Exception as e:
        print(f" Lỗi xử lý: {e}")
    finally:
        # Dọn dẹp file tạm sau khi dùng (tùy chọn)
        if os.path.exists(temp_path):
            pass # Để lại để debug nếu cần
            
    return None, None

In [ ]:
import requests
url_list = [
    "https://summerland.com.vn/wp-content/uploads/2025/11/huy-quan-hoa-la-ai-98ff77.webp",
    "https://summerland.com.vn/wp-content/uploads/2025/11/huy-quan-hoa-la-ai-59bd69.webp",
    "https://i.vgt.vn/2023/12/16/huy-quan-hoa-noi-len-nho-trinh-photoshop-dinh-cao-chim-sau-vi-lum-xum-cam-sung-vo-0f8-7049543.png",
    "https://i.vgt.vn/2023/12/16/huy-quan-hoa-noi-len-nho-trinh-photoshop-dinh-cao-chim-sau-vi-lum-xum-cam-sung-vo-92c-7049543.png",
    "https://i.vgt.vn/2023/12/16/huy-quan-hoa-noi-len-nho-trinh-photoshop-dinh-cao-chim-sau-vi-lum-xum-cam-sung-vo-d86-7049543.png",
    "https://i.vgt.vn/2023/12/16/huy-quan-hoa-noi-len-nho-trinh-photoshop-dinh-cao-chim-sau-vi-lum-xum-cam-sung-vo-fc6-7049543.png",
    "https://th.bing.com/th/id/R.67b0c537b4f69038b15ca12dbba5fee9?rik=kznJDh8PjvITdA&pid=ImgRaw&r=0",
    "https://5.pik.vn/a1f4d161d7053bab44634e8010de8aaa628f641fc6e3493f2ecbba0eb8fabb3e.webp"
]

# --- CHẠY KIỂM TRA TOÀN BỘ DANH SÁCH ---
for i, url in enumerate(url_list):
    img, prob = predict_external_image(url, loaded_model)
    
    if img is not None:
        plt.figure(figsize=(8, 5))
        plt.imshow(img)
        
        is_tampered = prob > 0.5
        label = "TAMPERED" if is_tampered else "AUTHENTIC"
        color = 'red' if is_tampered else 'green'
        
        plt.title(f"Mẫu số {i+1}\nKết luận: {label}\nConfidence Score: {prob:.4f}", 
                  fontsize=12, color=color, fontweight='bold')
        plt.axis('off')
        plt.show()